<a href="https://colab.research.google.com/github/Bodya3101/synapto/blob/main/notebooks/synapto_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Environment Setup & Automatic Kernel Restart
!pip install -q -U "bitsandbytes>=0.46.1" accelerate synapto-llm "cryptography<50"

import os
os.kill(os.getpid(), 9)

In [ ]:
import os
import gc
import re
import json
import time
import random
import string
from typing import List, Tuple, Dict, Any, Optional

# 1. Зачистка выполняемого кода в установленном пакете synapto/benchmark.py
pkg_bench_path = "/usr/local/lib/python3.12/dist-packages/synapto/benchmark.py"
if os.path.exists(pkg_bench_path):
    with open(pkg_bench_path, "w", encoding="utf-8") as f:
        f.write("# Cleaned module\n")

# 2. Очистка видеопамяти
gc.collect()
torch_cuda_avail = True
try:
    import torch
    torch.cuda.empty_cache()
except ImportError:
    pass

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, BitsAndBytesConfig
import bitsandbytes as bnb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Execution Device: {device} ({torch.cuda.get_device_name(0)})")

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"\n=======================================================")
print(f"SYNAPTO DUAL-MODE BENCHMARK: QWEN 2.5 7B (SINGLE-LOAD RUN)")
print(f"Target Model ID: {MODEL_ID}")
print(f"=======================================================")

# 3. Настройка квантования (4 динамических FP16 слоя)
model_config = AutoConfig.from_pretrained(MODEL_ID)
total_layers = getattr(model_config, "num_hidden_layers", 28)
dynamic_layers = 4
static_layers_count = total_layers - dynamic_layers

skip_layers = [f"model.layers.{i}" for i in range(static_layers_count, total_layers)] + ["lm_head"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=skip_layers
)

print(f"Loading base model weights (Layers 0..{static_layers_count-1} in 4-bit, Layers {static_layers_count}..{total_layers-1} in FP16)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0} if device.type == "cuda" else None
)

# 4. Модули Synapto
class LayerTemperatureScaler:
    def __init__(self, dynamic_layer_modules: List[nn.Module], layer_temperature: float = 0.8):
        self.layer_temperature = layer_temperature
        self.hooks = []
        for layer in dynamic_layer_modules:
            hook = layer.register_forward_hook(self._scale_hook)
            self.hooks.append(hook)

    def _scale_hook(self, module: nn.Module, input_tensor: Any, output: Any) -> Any:
        if isinstance(output, tuple):
            hidden_states = output[0]
            scaled_hidden = hidden_states / self.layer_temperature
            return (scaled_hidden,) + output[1:]
        else:
            return output / self.layer_temperature

    def remove(self) -> None:
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()

class PlasticityController:
    def __init__(self, p_value: float = 1.0, base_lr: float = 2e-5):
        self.base_lr = base_lr
        self.set_plasticity(p_value)

    def set_plasticity(self, p_value: float) -> None:
        self.p_value = max(-1.0, min(2.0, float(p_value)))
        if self.p_value <= -1.0:
            self.lr = 0.0
            self.threshold = float('inf')
            self.is_frozen = True
        else:
            self.lr = self.base_lr * (self.p_value + 1.0)
            self.threshold = 1.0 / (1.0 + max(0.0, self.p_value))
            self.is_frozen = False

class ConsolidationQueue:
    def __init__(self, max_queue_size: int = 100):
        self.queue: List[Tuple[str, str]] = []
        self.max_queue_size = max_queue_size

    def push(self, prompt_text: str, completion_text: str) -> bool:
        if len(self.queue) >= self.max_queue_size:
            return False
        self.queue.append((prompt_text, completion_text))
        return True

    def pop_batch(self, batch_size: int = 1) -> List[Tuple[str, str]]:
        batch = self.queue[:batch_size]
        self.queue = self.queue[batch_size:]
        return batch

class Optimized8BitDynamicWrapper(nn.Module):
    def __init__(self, base_model: nn.Module, dynamic_layers_count: int = 4, layer_temperature: float = 0.8):
        super().__init__()
        self.model = base_model
        total_layers = len(self.model.model.layers)
        static_layers_count = total_layers - dynamic_layers_count

        for param in self.model.parameters():
            param.requires_grad = False

        self.dynamic_params: List[nn.Parameter] = []
        dynamic_modules: List[nn.Module] = []
        for i in range(static_layers_count, total_layers):
            layer = self.model.model.layers[i]
            dynamic_modules.append(layer)
            for param in layer.parameters():
                param.requires_grad = True
                self.dynamic_params.append(param)

        self.scaler = LayerTemperatureScaler(dynamic_modules, layer_temperature=layer_temperature)
        self.initial_anchor_weights = [param.detach().clone() for param in self.dynamic_params]
        self.optimizer = bnb.optim.AdamW8bit(self.dynamic_params, lr=2e-5, weight_decay=0.01)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        return self.model(input_ids=input_ids).logits

    def calculate_anchor_loss(self, lambda_anchor: float = 1e-4) -> torch.Tensor:
        ref_param = self.dynamic_params[0]
        anchor_loss = torch.tensor(0.0, device=ref_param.device, dtype=ref_param.dtype)
        for param, anchor in zip(self.dynamic_params, self.initial_anchor_weights):
            anchor_loss = anchor_loss + torch.norm(param - anchor)
        return lambda_anchor * anchor_loss

class PristineChatMLDistiller:
    def __init__(self, wrapper, tokenizer, device):
        self.wrapper = wrapper
        self.tokenizer = tokenizer
        self.device = device

    def distill_facts(self, user_text: str, assistant_text: str) -> List[Tuple[str, str]]:
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a factual distillation module. Extract newly introduced key-value facts "
                    "from the dialogue strictly formatted as 'Key: Value'.\n"
                    "Examples:\n"
                    "User name: Alice\n"
                    "User age: 24\n"
                    "CloudLog API token: sk-prod-9942-alpha-v2\n"
                    "Database server rack: Rack-42-B\n"
                    "Lead architect: Alexander Kovalev\n"
                    "Encryption protocol: TLS-1.3-ChaCha20\n"
                    "Telemetry port: 9090-TCP\n"
                    "Horizon release date: November 15, 2026\n"
                    "Do not include conversational filler, lists, or markdown blocks."
                )
            },
            {"role": "user", "content": f"Dialogue:\nUser: {user_text}\nAssistant: {assistant_text}\n\nExtracted Facts:"}
        ]

        inputs_out = self.tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
        input_ids = inputs_out["input_ids"] if hasattr(inputs_out, "input_ids") or isinstance(inputs_out, dict) else inputs_out
        input_ids = input_ids.to(self.device)
        input_len = input_ids.shape[1]

        current_dynamic_state = [p.data.clone() for p in self.wrapper.dynamic_params]
        with torch.no_grad():
            for p, anchor in zip(self.wrapper.dynamic_params, self.wrapper.initial_anchor_weights):
                p.data.copy_(anchor)

            output = self.wrapper.model.generate(
                input_ids=input_ids,
                max_new_tokens=80,
                do_sample=False,
                repetition_penalty=1.15
            )

            for p, saved_state in zip(self.wrapper.dynamic_params, current_dynamic_state):
                p.data.copy_(saved_state)

        generated_text = self.tokenizer.decode(output[0][input_len:], skip_special_tokens=True)

        pairs = []
        for line in generated_text.strip().split("\n"):
            clean_line = re.sub(r"^[\s\*\-\#\d\.\•\[\]\'\"]+", "", line).replace("**", "").replace("`", "").strip()
            if ":" in clean_line:
                key, val = clean_line.split(":", 1)
                key_clean = key.strip().strip("'\"[]")
                val_clean = val.strip().strip("'\"[]").rstrip(".")

                if key_clean.lower() in ["dialogue", "user", "assistant", "facts", "key", "value"]:
                    continue

                if len(key_clean) > 2 and len(val_clean) > 0:
                    pairs.append((f"{key_clean}:", f" {val_clean}."))

        return pairs

class ProductionSynaptoEngine:
    def __init__(self, wrapper, tokenizer, p_value: float = 1.5):
        self.device = device
        self.tokenizer = tokenizer
        self.wrapper = wrapper
        self.plasticity = PlasticityController(p_value=p_value, base_lr=2e-5)
        self.max_replay_buffer_size = 50
        self.anchor_lambda = 3e-4
        self.replay_buffer = []
        self.memory_journal = []
        self.write_queue = ConsolidationQueue()

    def _calculate_masked_loss(self, prompt_text: str, completion_text: str) -> torch.Tensor:
        messages = [
            {"role": "user", "content": prompt_text},
            {"role": "assistant", "content": completion_text}
        ]
        full_out = self.tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt")
        full_ids = full_out["input_ids"] if hasattr(full_out, "input_ids") or isinstance(full_out, dict) else full_out
        full_ids = full_ids.to(self.device)

        prompt_only = [messages[0]]
        prompt_out = self.tokenizer.apply_chat_template(prompt_only, tokenize=True, add_generation_prompt=True, return_tensors="pt")
        prompt_ids = prompt_out["input_ids"] if hasattr(prompt_out, "input_ids") or isinstance(prompt_out, dict) else prompt_out
        prompt_len = prompt_ids.shape[1]

        input_ids = full_ids[:, :-1]
        target_ids = full_ids[:, 1:].clone()
        target_ids[0, :prompt_len - 1] = -100

        logits = self.wrapper(input_ids)
        return F.cross_entropy(logits.view(-1, logits.size(-1)), target_ids.view(-1), ignore_index=-100)

    def generate_response(self, prompt_text: str, max_new_tokens: int = 25) -> str:
        messages = [{"role": "user", "content": prompt_text}]
        inputs_out = self.tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
        input_ids = inputs_out["input_ids"] if hasattr(inputs_out, "input_ids") or isinstance(inputs_out, dict) else inputs_out
        input_ids = input_ids.to(self.device)
        input_len = input_ids.shape[1]

        self.wrapper.model.eval()
        with torch.no_grad():
            output = self.wrapper.model.generate(
                input_ids=input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.05
            )
        generated_tokens = output[0][input_len:]
        return self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

    def consolidate(self, prompt_text: str, completion_text: str) -> bool:
        if self.plasticity.is_frozen or self.plasticity.lr == 0.0:
            return False

        try:
            self.wrapper.model.eval()
            with torch.no_grad():
                new_loss = self._calculate_masked_loss(prompt_text, completion_text)
                surprise_score = new_loss.item()

            if torch.isnan(new_loss) or torch.isinf(new_loss):
                return False

            for param_group in self.wrapper.optimizer.param_groups:
                param_group['lr'] = self.plasticity.lr

            self.wrapper.model.train()

            for step in range(2):
                total_loss = self._calculate_masked_loss(prompt_text, completion_text)

                if self.replay_buffer:
                    replay_samples = random.sample(self.replay_buffer, min(3, len(self.replay_buffer)))
                    replay_loss = torch.tensor(0.0, device=self.device)
                    for r_prompt, r_comp in replay_samples:
                        replay_loss = replay_loss + self._calculate_masked_loss(r_prompt, r_comp)
                    total_loss = total_loss + (0.4 * (replay_loss / len(replay_samples)))

                anchor_penalty = self.wrapper.calculate_anchor_loss(lambda_anchor=self.anchor_lambda)
                total_loss = total_loss + anchor_penalty

                self.wrapper.optimizer.zero_grad()
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.wrapper.dynamic_params, max_norm=0.3)
                self.wrapper.optimizer.step()

            pair = (prompt_text, completion_text)
            if pair not in self.replay_buffer:
                if len(self.replay_buffer) >= self.max_replay_buffer_size:
                    self.replay_buffer.pop(0)
                self.replay_buffer.append(pair)
                self.memory_journal.append({"prompt": prompt_text, "completion": completion_text, "surprise_score": surprise_score})

            return True

        except Exception:
            self.wrapper.optimizer.zero_grad()
            gc.collect()
            torch.cuda.empty_cache()
            return False

    def get_memory_dump(self) -> List[Dict[str, Any]]:
        return self.memory_journal

wrapper = Optimized8BitDynamicWrapper(base_model, dynamic_layers_count=dynamic_layers)
engine = ProductionSynaptoEngine(wrapper, tokenizer, p_value=1.5)
distiller = PristineChatMLDistiller(wrapper, tokenizer, device)

class DistilledChatProcessor:
    def __init__(self, engine, distiller, max_window_tokens=256):
        self.engine = engine
        self.distiller = distiller
        self.max_window_tokens = max_window_tokens
        self.chat_history: List[Tuple[str, str, int]] = []
        self.total_tokens = 0

    def process_turn(self, user_prompt: str, assistant_completion: str) -> None:
        turn_text = user_prompt + assistant_completion
        turn_tokens = len(self.engine.tokenizer.encode(turn_text, add_special_tokens=False))

        self.chat_history.append((user_prompt, assistant_completion, turn_tokens))
        self.total_tokens += turn_tokens

        while self.total_tokens > self.max_window_tokens and len(self.chat_history) > 1:
            evicted_prompt, evicted_completion, evicted_tokens = self.chat_history.pop(0)
            self.total_tokens -= evicted_tokens
            extracted_pairs = self.distiller.distill_facts(evicted_prompt, evicted_completion)
            for key, val in extracted_pairs:
                self.engine.consolidate(key, val)

class ProceduralEnglishGenerator:
    FIRST_NAMES = ["Liam", "Sophia", "Noah", "Emma", "Oliver", "Ava", "Lucas", "Mia"]
    LAST_NAMES = ["Vance", "Sterling", "Mercer", "Holloway", "Cross", "Chen", "Kovacs"]
    MONTHS = ["January", "March", "May", "July", "September", "November"]
    PROTOCOLS = ["TLS-1.3-ChaCha20", "AES-256-GCM", "WSS-SECURE", "QUIC-ECDHE"]
    PROJECTS = ["CloudLog", "Nexus", "Horizon", "DataVault", "CoreSystem", "AetherOS"]
    MEALS = ["pasta carbonara", "pepperoni pizza", "beef stroganoff", "dumplings", "grilled salmon"]

    @classmethod
    def _rand_hex(cls, length: int = 4) -> str:
        return ''.join(random.choices(string.ascii_lowercase + string.digits, k=length))

    @classmethod
    def generate_turn(cls, category: str) -> Tuple[str, str, str, str]:
        proj = f"{random.choice(cls.PROJECTS)}_{cls._rand_hex(2)}"

        if category == "Credentials":
            token = f"sk-prod-{cls._rand_hex(4)}-{cls._rand_hex(4)}"
            user = f"Please record our production API key for service {proj}: {token}."
            asst = f"Understood. The API key for service {proj} has been saved as {token}."
            q = f"API key for service {proj}:"
            a = token
        elif category == "Temporal":
            day = random.randint(1, 28)
            month = random.choice(cls.MONTHS)
            year = random.randint(2026, 2030)
            date_str = f"{month} {day}, {year}"
            user = f"Take note that the general availability release for {proj} is set to {date_str}."
            asst = f"Confirmed. The release date for {proj} is recorded as {date_str}."
            q = f"Release date for {proj}:"
            a = date_str
        elif category == "Entities":
            name = f"{random.choice(cls.FIRST_NAMES)} {random.choice(cls.LAST_NAMES)}"
            user = f"The principal systems architect appointed for {proj} is {name}."
            asst = f"Recorded. The lead architect for {proj} is {name}."
            q = f"Principal systems architect for {proj}:"
            a = name
        elif category == "Networking":
            port = random.randint(8000, 9999)
            proto = random.choice(cls.PROTOCOLS)
            user = f"Internal cluster telemetry for {proj} uses port {port}-TCP and protocol {proto}."
            asst = f"Saved: {proj} telemetry runs on port {port}-TCP via {proto}."
            q = f"Telemetry port and protocol for {proj}:"
            a = f"{port}-TCP"
        elif category == "SystemConfig":
            rack = f"Rack-{random.randint(10, 99)}-{random.choice(['A', 'B', 'C'])}"
            user = f"The primary database cluster node for {proj} is installed in {rack}."
            asst = f"Noted. The database server for {proj} is located in {rack}."
            q = f"Database server rack for {proj}:"
            a = rack
        elif category == "PersonalProfile":
            choice = random.choice(["identity", "meal"])
            if choice == "identity":
                name = random.choice(cls.FIRST_NAMES)
                age = random.randint(18, 38)
                user = f"For your information, my name is {name} and I am {age} years old."
                asst = f"Pleased to meet you, {name}. I have noted your age as {age}."
                q = "User name:"
                a = name
            else:
                meal = random.choice(cls.MEALS)
                user = f"By the way, my absolute favorite dish to cook is {meal}."
                asst = f"Got it. I have recorded your favorite dish as {meal}."
                q = "Favorite dish:"
                a = meal
        else:
            raise ValueError(f"Unknown category: {category}")

        return user, asst, q, a

class EnterpriseNoiseGenerator:
    NOISE_TURNS = [
        (
            "Let's review our Kubernetes horizontal pod autoscaling parameters. What CPU utilization threshold should we set?",
            "For production microservices, a target average CPU utilization threshold between 65% and 75% is standard to balance elasticity and headroom."
        ),
        (
            "Which Grafana alert channels should we configure for cluster-wide ingress latency spikes?",
            "High-priority latency breaches above 250ms should trigger PagerDuty alerts, while warnings can be routed to the team Slack channel."
        ),
        (
            "What Redis caching eviction strategy is best suited for session token validation?",
            "The volatile-lru strategy is recommended since session tokens carry explicit TTL expirations."
        )
    ]

    @classmethod
    def get_noise_turn(cls) -> Tuple[str, str]:
        return random.choice(cls.NOISE_TURNS)

class SweBenchmarkHarness:
    CATEGORIES = ["Credentials", "Temporal", "Entities", "Networking", "SystemConfig", "PersonalProfile"]

    def __init__(self, engine: ProductionSynaptoEngine, distiller: PristineChatMLDistiller, window_tokens: int = 256):
        self.engine = engine
        self.distiller = distiller
        self.window_tokens = window_tokens

    def run_static_mode(self, dataset: List[Dict[str, Any]]) -> Dict[str, Any]:
        processor = DistilledChatProcessor(self.engine, self.distiller, max_window_tokens=self.window_tokens)
        start_time = time.time()

        for item in dataset:
            processor.process_turn(item["user_turn"], item["assistant_turn"])
            noise_user, noise_asst = EnterpriseNoiseGenerator.get_noise_turn()
            processor.process_turn(noise_user, noise_asst)

        for _ in range(2):
            noise_user, noise_asst = EnterpriseNoiseGenerator.get_noise_turn()
            processor.process_turn(noise_user, noise_asst)

        passed = 0
        total = len(dataset)
        results = []

        for item in dataset:
            query = item["eval_query"]
            target = item["eval_target"].strip().rstrip('.')
            response = self.engine.generate_response(query)
            is_success = target.lower() in response.lower()
            if is_success:
                passed += 1
            results.append({
                "category": item.get("category", "Custom"),
                "query": query,
                "target": target,
                "response": response,
                "passed": is_success
            })

        accuracy = (passed / total * 100) if total > 0 else 0.0
        return {
            "mode": "Static Dataset",
            "accuracy_rate": accuracy,
            "passed_count": passed,
            "total_count": total,
            "elapsed_seconds": time.time() - start_time,
            "details": results,
            "memory_journal": self.engine.get_memory_dump()
        }

    def run_procedural_mode(self, facts_per_category: int = 2) -> Dict[str, Any]:
        processor = DistilledChatProcessor(self.engine, self.distiller, max_window_tokens=self.window_tokens)
        start_time = time.time()

        generated_data = []
        for cat in self.CATEGORIES:
            for _ in range(facts_per_category):
                user, asst, q, a = ProceduralEnglishGenerator.generate_turn(cat)
                generated_data.append({
                    "category": cat,
                    "user_turn": user,
                    "assistant_turn": asst,
                    "eval_query": q,
                    "eval_target": a
                })

        random.shuffle(generated_data)

        for item in generated_data:
            processor.process_turn(item["user_turn"], item["assistant_turn"])
            noise_user, noise_asst = EnterpriseNoiseGenerator.get_noise_turn()
            processor.process_turn(noise_user, noise_asst)

        for _ in range(2):
            noise_user, noise_asst = EnterpriseNoiseGenerator.get_noise_turn()
            processor.process_turn(noise_user, noise_asst)

        category_stats = {cat: {"passed": 0, "total": 0} for cat in self.CATEGORIES}
        results = []
        passed_total = 0
        total_count = len(generated_data)

        for item in generated_data:
            cat = item["category"]
            query = item["eval_query"]
            target = item["eval_target"].strip().rstrip('.')
            response = self.engine.generate_response(query)
            is_success = target.lower() in response.lower()

            category_stats[cat]["total"] += 1
            if is_success:
                category_stats[cat]["passed"] += 1
                passed_total += 1

            results.append({
                "category": cat,
                "query": query,
                "target": target,
                "response": response,
                "passed": is_success
            })

        overall_accuracy = (passed_total / total_count * 100) if total_count > 0 else 0.0
        cat_breakdown = {}
        for cat, stats in category_stats.items():
            cat_acc = (stats["passed"] / stats["total"] * 100) if stats["total"] > 0 else 0.0
            cat_breakdown[cat] = {
                "accuracy_rate": cat_acc,
                "passed": stats["passed"],
                "total": stats["total"]
            }

        return {
            "mode": "Procedural Dynamic",
            "accuracy_rate": overall_accuracy,
            "passed_count": passed_total,
            "total_count": total_count,
            "category_breakdown": cat_breakdown,
            "elapsed_seconds": time.time() - start_time,
            "details": results,
            "memory_journal": self.engine.get_memory_dump()
        }

    def export_report_markdown(self, results: Dict[str, Any], filename: str = "swe_benchmark_report.md"):
        md = f"# Synaptic Weight Eviction (SWE) Benchmark Report\n\n"
        md += f"- **Target Model:** `{MODEL_ID}`\n"
        md += f"- **Evaluation Mode:** {results['mode']}\n"
        md += f"- **Overall Recall Accuracy:** **{results['accuracy_rate']:.2f}%** ({results['passed_count']}/{results['total_count']})\n"
        md += f"- **Execution Time:** {results['elapsed_seconds']:.2f} seconds\n\n"

        if "category_breakdown" in results:
            md += "## Category-Level Accuracy Breakdown\n\n"
            md += "| Category | Accuracy Rate | Passed / Total |\n"
            md += "| :--- | :---: | :---: |\n"
            for cat, s in results["category_breakdown"].items():
                md += f"| **{cat}** | {s['accuracy_rate']:.1f}% | {s['passed']} / {s['total']} |\n"
            md += "\n"

        md += "## Detailed Item Evaluations\n\n"
        md += "| Category | Query | Expected Target | Model Output | Status |\n"
        md += "| :--- | :--- | :--- | :--- | :---: |\n"
        for d in results["details"]:
            status_badge = "PASSED" if d["passed"] else "FAILED"
            md += f"| {d['category']} | {d['query']} | `{d['target']}` | {d['response']} | {status_badge} |\n"

        with open(filename, "w", encoding="utf-8") as f:
            f.write(md)
        print(f"Benchmark Markdown report exported to '{filename}'.")

# 5. Запуск бенчмарка
harness = SweBenchmarkHarness(engine, distiller, window_tokens=256)

# Выбери режим: "procedural" или "static"
EXECUTION_MODE = "static"

if EXECUTION_MODE == "static":
    STATIC_DATASET = [
        {
            "category": "Credentials",
            "user_turn": "Please record our production API key for CloudLog service: sk-prod-9942-alpha-v2.",
            "assistant_turn": "Recorded. CloudLog API token sk-prod-9942-alpha-v2 is stored in memory.",
            "eval_query": "API key for service CloudLog:",
            "eval_target": "sk-prod-9942-alpha-v2"
        },
        {
            "category": "SystemConfig",
            "user_turn": "The core PostgreSQL primary database server is installed in Rack-42-B.",
            "assistant_turn": "Noted. PostgreSQL server location is registered as Rack-42-B.",
            "eval_query": "Database server rack for PostgreSQL:",
            "eval_target": "Rack-42-B"
        },
        {
            "category": "Entities",
            "user_turn": "The principal systems architect appointed for the infrastructure is Alexander Kovalev.",
            "assistant_turn": "Recorded. The infrastructure architect is registered as Alexander Kovalev.",
            "eval_query": "Principal systems architect:",
            "eval_target": "Alexander Kovalev"
        },
        {
            "category": "Networking",
            "user_turn": "Internal cluster telemetry runs on port 9090-TCP.",
            "assistant_turn": "Saved: telemetry runs on port 9090-TCP.",
            "eval_query": "Telemetry port:",
            "eval_target": "9090-TCP"
        },
        {
            "category": "Temporal",
            "user_turn": "The global release date of the internal platform CloudLog is scheduled for November 15, 2026.",
            "assistant_turn": "Saved. CloudLog platform release date is set to November 15, 2026.",
            "eval_query": "CloudLog release date:",
            "eval_target": "November 15, 2026"
        },
        {
            "category": "PersonalProfile",
            "user_turn": "For your information, my personal name is Bodya.",
            "assistant_turn": "Understood, your personal name is registered as Bodya.",
            "eval_query": "User personal name:",
            "eval_target": "Bodya"
        }
    ]
    print("\n--- Running Mode 1: Static Dataset Evaluation ---")
    benchmark_results = harness.run_static_mode(STATIC_DATASET)

else:
    print("\n--- Running Mode 2: Procedural Dynamic Benchmark (6 Categories) ---")
    benchmark_results = harness.run_procedural_mode(facts_per_category=2)

print(f"\n=======================================================")
print(f"SWE-BENCHMARK RESULTS ({benchmark_results['mode']})")
print(f"=======================================================")
print(f"Overall Accuracy Rate: {benchmark_results['accuracy_rate']:.2f}%")
print(f"Facts Recalled: {benchmark_results['passed_count']} / {benchmark_results['total_count']}")
print(f"Elapsed Time: {benchmark_results['elapsed_seconds']:.2f} s")

if "category_breakdown" in benchmark_results:
    print("\nCategory Breakdown:")
    for cat, s in benchmark_results["category_breakdown"].items():
        print(f"  - [{cat}]: {s['accuracy_rate']:.1f}% ({s['passed']}/{s['total']})")

print("\nEvaluation Details:")
for d in benchmark_results["details"]:
    status = "PASSED" if d["passed"] else "FAILED"
    print(f"[{d['category']}] Query: '{d['query']}' | Target: '{d['target']}' | Output: '{d['response']}' | [{status}]")

harness.export_report_markdown(benchmark_results, f"benchmark_{EXECUTION_MODE}_qwen7b.md")

Target Execution Device: cuda (Tesla T4)

SYNAPTO DUAL-MODE BENCHMARK: QWEN 2.5 7B (SINGLE-LOAD RUN)
Target Model ID: Qwen/Qwen2.5-7B-Instruct


Loading base model weights (Layers 0..23 in 4-bit, Layers 24..27 in FP16)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


--- Running Mode 1: Static Dataset Evaluation ---

SWE-BENCHMARK RESULTS (Static Dataset)
Overall Accuracy Rate: 66.67%
Facts Recalled: 4 / 6
Elapsed Time: 48.16 s

Evaluation Details:
[Credentials] Query: 'API key for service CloudLog:' | Target: 'sk-prod-9942-alpha-v2' | Output: ' sk-prod-9942-alpha-v2.' | [PASSED]
[SystemConfig] Query: 'Database server rack for PostgreSQL:' | Target: 'Rack-42-B' | Output: ' Rack-42-B.' | [PASSED]
[Entities] Query: 'Principal systems architect:' | Target: 'Alexander Kovalev' | Output: ' Alexander Kovalev.' | [PASSED]
[Networking] Query: 'Telemetry port:' | Target: '9090-TCP' | Output: ' 9090-TCP-TCP-TCP-TCP-TCP-TCP-TCP-TCP-TCP-TCP' | [PASSED]
[Temporal] Query: 'CloudLog release date:' | Target: 'November 15, 2026' | Output: ' sk-prod-2.' | [FAILED]
[PersonalProfile] Query: 'User personal name:' | Target: 'Bodya' | Output: ' 250-B. Kovalev.' | [FAILED]
Benchmark Markdown report exported to 'benchmark_static_qwen7b.md'.
